In [57]:
import nltk
nltk.download('plunkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Error loading plunkt: Package 'plunkt' not found in index
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [58]:
"""import pandas as pd
df = pd.read_csv(r'D:\natural_language_processing\dataset\spam.csv' ,sep='\t',header=None, names=['label','message'],encoding='latin-1')
print(df.shape)
print(df['label'].value_counts())
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print(df.head(3))"""
import pandas as pd

# sep=',' use karo
df = pd.read_csv(r'D:\natural_language_processing\dataset\spam.csv', 
                 encoding='latin-1',
                 sep=',',
                 on_bad_lines='skip')   # broken rows skip karo

# Sirf pehle 2 columns rakho
df = df.iloc[:, :2]
df.columns = ['label', 'message']

# NaN hatao aur string banao
df = df.dropna(subset=['message'])
df['message'] = df['message'].astype(str)

# Sirf ham aur spam rows rakho
df = df[df['label'].isin(['ham', 'spam'])]

print(df.shape)
print(df['label'].value_counts())
print(df.head(3))

<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\lenovo\AppData\Local\Temp\ipykernel_26552\2701146689.py:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df = pd.read_csv(r'D:\natural_language_processing\dataset\spam.csv' ,sep='\t',header=None, names=['label','message'],encoding='latin-1')


(5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...


In [59]:
##df = df.dropna(subset=['message'])
##df['message'] = df['message'].fillna('')
df = df.dropna(subset=['message'])
df['message'] = df['message'].fillna('').astype(str) 


In [60]:
from sklearn.model_selection import train_test_split
df['label_encoded'] = (df['label']=='spam').astype(int)
X_raw = df['message']
Y=df['label_encoded']
X_train_raw ,X_test_raw ,Y_train ,Y_test =train_test_split(X_raw ,Y,test_size=0.2,random_state=42,stratify=Y)

In [61]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', 'url', text)
    text = re.sub(r'\d+', 'num', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [62]:
from nltk.tokenize import word_tokenize
def tokenize(text):
    return word_tokenize(text)

In [63]:
from nltk.corpus import stopwords
stop_words=set(stopwords.words('english'))
def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]  

In [64]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
def lemmatize_tokens(tokens) :
    return [lemmatizer.lemmatize(token) for token in tokens]    

In [65]:
def preprocess(text):
    text   = clean_text(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize_tokens(tokens)
    return ' '.join(tokens)

X_train = X_train_raw.apply(preprocess)
X_test  = X_test_raw.apply(preprocess)

In [66]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vec = CountVectorizer(max_features=2500, ngram_range=(1,1), min_df=2)

X_train_bow = bow_vec.fit_transform(X_train)   
X_test_bow  = bow_vec.transform(X_test)         

In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer(max_features=2500, ngram_range=(1,1), sublinear_tf=True)

X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf  = tfidf_vec.transform(X_test)

In [68]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, Y_train)

y_pred = lr.predict(X_test_tfidf)
print(classification_report(Y_test, y_pred, target_names=['ham', 'spam']))

              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       966
        spam       0.95      0.86      0.90       149

    accuracy                           0.97      1115
   macro avg       0.96      0.93      0.94      1115
weighted avg       0.97      0.97      0.97      1115

